In [18]:
import sonnet as snt
import tensorflow as tf
import numpy as np

In [19]:
from migration.datasets import create_AIS_dataset
inputs, targets, _, _, _, lengths, mean =  create_AIS_dataset('../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl', 
                   '../../data/ct_2017010203_10_20/mean.pkl',
                   32,
                   99999, # not used lol
                   300,
                   300, 
                   30,
                   72, 
                   shuffle=False,
                   repeat=False)

In [20]:
latent_size = 64
_DEFAULT_INITIALIZERS = {"w": tf.compat.v1.keras.initializers.VarianceScaling(scale=1.0, mode="fan_avg", distribution="uniform",seed=111),
                         "b": tf.compat.v1.zeros_initializer()}

weights sampled from [-limit, limit], where limit = sqrt(3 * scale / n) and n = (n_units_in + n_units_out)/2

Initializers: expect shapes

In [21]:

with tf.Session() as sess:
    w_init = sess.run(_DEFAULT_INITIALIZERS['w'](shape=(2,3)))

In [22]:
w_init

array([[ 0.78979087,  0.48955142, -0.4669335 ],
       [-0.0235517 ,  0.89863634,  0.6412705 ]], dtype=float32)

In [23]:
with tf.Session() as sess:
    b_init = sess.run(_DEFAULT_INITIALIZERS['b'](shape=(2,3)))

In [24]:
b_init

array([[0., 0., 0.],
       [0., 0., 0.]], dtype=float32)

In [25]:
data_feat_extractor = snt.nets.MLP(
      output_sizes=[latent_size, latent_size],
      initializers=_DEFAULT_INITIALIZERS,
      name="data_feat_extractor")

In [26]:
inputs_encoded = data_feat_extractor(inputs[1])

In [27]:
data_feat_extractor.get_all_variables()

(<tf.Variable 'data_feat_extractor_1/linear_0/b:0' shape=(64,) dtype=float32_ref>,
 <tf.Variable 'data_feat_extractor_1/linear_0/w:0' shape=(702, 64) dtype=float32_ref>,
 <tf.Variable 'data_feat_extractor_1/linear_1/b:0' shape=(64,) dtype=float32_ref>,
 <tf.Variable 'data_feat_extractor_1/linear_1/w:0' shape=(64, 64) dtype=float32_ref>)

In [28]:
data_feat_extractor.initializers

{'w': <tensorflow.python.ops.init_ops.VarianceScaling at 0x7638725dd290>,
 'b': <tensorflow.python.ops.init_ops.Zeros at 0x7638725dd350>}

In [29]:
tf.compat.v1.global_variables()

[<tf.Variable 'data_feat_extractor/linear_0/w:0' shape=(702, 64) dtype=float32_ref>,
 <tf.Variable 'data_feat_extractor/linear_0/b:0' shape=(64,) dtype=float32_ref>,
 <tf.Variable 'data_feat_extractor/linear_1/w:0' shape=(64, 64) dtype=float32_ref>,
 <tf.Variable 'data_feat_extractor/linear_1/b:0' shape=(64,) dtype=float32_ref>,
 <tf.Variable 'data_feat_extractor_1/linear_0/w:0' shape=(702, 64) dtype=float32_ref>,
 <tf.Variable 'data_feat_extractor_1/linear_0/b:0' shape=(64,) dtype=float32_ref>,
 <tf.Variable 'data_feat_extractor_1/linear_1/w:0' shape=(64, 64) dtype=float32_ref>,
 <tf.Variable 'data_feat_extractor_1/linear_1/b:0' shape=(64,) dtype=float32_ref>]

In [30]:
with tf.Session() as sess:
    init = tf.compat.v1.global_variables_initializer()
    sess.run(init)
    inputs_encoded = sess.run(inputs_encoded)

In [31]:
with tf.Session() as sess:
    inputs_0 = sess.run(inputs[1])
np.unique(inputs_0)

array([0., 1.], dtype=float32)

In [32]:
inputs_encoded

array([[ 0.04229711, -0.11309095, -0.03396244, ...,  0.02506403,
        -0.05846981,  0.05129394],
       [ 0.01303088, -0.12961668,  0.00366738, ..., -0.01928811,
        -0.04857266,  0.08521809],
       [ 0.07504959, -0.1725211 , -0.08074127, ..., -0.01611277,
         0.07610114,  0.02177772],
       ...,
       [ 0.21280336, -0.09041511, -0.05578803, ..., -0.08589473,
        -0.10923211,  0.08138398],
       [ 0.04365487, -0.1192705 , -0.06853262, ...,  0.00054709,
         0.0023195 ,  0.01772047],
       [ 0.05795524, -0.11295623, -0.0736052 , ..., -0.10135509,
         0.09828771,  0.11088958]], dtype=float32)